<a href="https://colab.research.google.com/github/robertbarcik/ADK-tutorial/blob/main/notebooks/13_live_api_voice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **Needs a Google API key.** The bidirectional Live (voice) API is a native Gemini feature, so this notebook calls `google.genai` directly rather than routing through OpenRouter. It ships without stored outputs because it cannot run on the OpenRouter-only setup used for the rest of the course. To run it, open it in Colab (badge above), set a `GOOGLE_API_KEY` (from Google AI Studio) in the environment or Colab secrets, and run the cells.

# Module 13 — Live API Voice Agent

The third Gemini-only unlock. The biggest one.

**The Live API lets Gemini exchange audio bidirectionally with the user in real time.** Voice in, voice out. Voice activity detection. User-interruption-aware. Sub-second latency on happy paths. Nothing in the Claude, GPT, or open-weight ecosystem ships this cleanly as of mid-2026. OpenAI's Realtime API comes closest but has different semantics, different pricing, and different failure modes.

The production use cases: voice assistants, live interpreter agents, real-time coaching, accessibility tools, voice-driven customer support.

**Fair warning — Live is the most fragile of the three Gemini unlocks.** The API is preview-tier and the model names change frequently. Through spring 2026 the endpoint threw transient server-side 1011 errors on the free tier; as of July 2026 sessions connect reliably again, but treat any single failure as the endpoint's mood, not your code. The course repo's `DEMOS_BROKEN.md` tracks the current state.

This module teaches the **mechanics** — the API shape, the queue-based interaction pattern, what's special about it. The concepts are more durable than this month's version-string.

**What you'll leave with:**
- The `run_live()` and `LiveRequestQueue` API and how it differs from `run_async()`.
- Voice activity detection + interruption handling — what they do and where they live in the API.
- Knowing which model names actually ship the Live capability (moving target; check catalog).
- A production architecture sketch: browser microphone → WebSocket → ADK Live agent → audio out.

**Running cost:** Live audio is ~$0.012 per minute on Gemini Flash Live (preview tier). Short demos < $0.01.

# Setup

In [ ]:
!pip install -q google-adk==2.4.0 google-genai litellm==1.85.7 python-dotenv==1.0.1 nest-asyncio==1.6.0 deprecated==1.2.18 2>/dev/null
print("✅ Packages installed.")

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")
try: sys.stderr.fileno()
except Exception: sys.stderr = open(os.devnull, "w")

GOOGLE_API_KEY = None
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    try:
        from dotenv import load_dotenv; load_dotenv()
        GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
    except ImportError: pass
if not GOOGLE_API_KEY:
    from getpass import getpass
    GOOGLE_API_KEY = getpass("Enter your Google AI Studio API key: ")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"
print("✅ Environment ready.")

In [ ]:
import asyncio, logging
import nest_asyncio; nest_asyncio.apply()
logging.getLogger("LiteLLM").setLevel(logging.WARNING)

from google.adk.agents import LlmAgent
from google.adk.agents.run_config import RunConfig
from google.adk.agents.live_request_queue import LiveRequestQueue
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google import genai
from google.genai import types

print("✅ Imports successful.")

# What's Different About Live

Every module until now used `runner.run_async(new_message=...)` — a **request/response** cycle. You hand ADK a user message, ADK drives the agent, events stream back, the turn ends, you move on.

Live is different. The agent holds **an open bidirectional WebSocket** to Gemini. You don't hand over a single message; you stream chunks of audio (or text) into a queue, and chunks of response audio (or text) stream back, in parallel, for as long as the session is open.

Three primitives you need to know about:

| Primitive | Role |
|---|---|
| `LiveRequestQueue` | The client-side queue you push user input (text chunks, audio blobs) into |
| `Runner.run_live(...)` | The async generator that yields server events; runs until you close the queue |
| `RunConfig(response_modalities=[...])` | Output modality. Current live models are audio-native — request `["AUDIO"]`; a TEXT-only request is rejected with error 1007. Add `output_audio_transcription` to stream the words too |

The model names that ship the Live capability move around — check [ai.google.dev/gemini-api/docs/live](https://ai.google.dev/gemini-api/docs/live) for the current list. As of this writing, `gemini-3.1-flash-live-preview` is the main free-tier-available Live model; `gemini-live-2.5-flash-native-audio` is production-grade but paid tier.

# Discover the Available Live Models

In [ ]:
# Enumerate which models on your key actually support bidirectional live content.
client = genai.Client(api_key=GOOGLE_API_KEY)

print("Live-capable models on this API key:")
any_found = False
for m in client.models.list():
    actions = getattr(m, "supported_actions", None) or []
    if "bidiGenerateContent" in actions:
        any_found = True
        print(f"  {m.name}")

if not any_found:
    print("  (none listed — Live may not be enabled on this key)")

# The API Shape — One Live Audio Turn

The demo below runs a single Live turn end to end: one typed prompt goes into the queue, spoken audio comes back, and the words arrive alongside it via output transcription.

Two things to notice before running it. Current live models are **audio-native** — ask for `response_modalities=["TEXT"]` and the server rejects the session with error 1007. So we request `AUDIO` and attach `output_audio_transcription`, which streams the transcript of what the model is saying next to the audio bytes.

The Live API is still preview-tier. If the cell fails with a transient server error, re-run it — and see `DEMOS_BROKEN.md` in the repo for the current state.


In [ ]:
LIVE_MODEL = "gemini-3.1-flash-live-preview"

live_agent = LlmAgent(
    name="live_chat_agent",
    model=LIVE_MODEL,
    description="A conversational live-audio agent.",
    instruction=(
        "You chat casually with the user. Keep replies short and natural, "
        "like a quick spoken exchange. No markdown."
    ),
)

APP = "m13_live"
USER = "student"
session_service = InMemorySessionService()

async def live_audio_turn(prompt: str):
    await session_service.create_session(app_name=APP, user_id=USER, session_id="live1")
    runner = Runner(agent=live_agent, app_name=APP, session_service=session_service)

    queue = LiveRequestQueue()
    # Live models are audio-native: audio out, plus transcription so we can read it.
    config = RunConfig(
        response_modalities=["AUDIO"],
        output_audio_transcription=types.AudioTranscriptionConfig(),
    )

    # Feed one turn of user content into the queue
    queue.send_content(
        content=types.Content(role="user", parts=[types.Part(text=prompt)])
    )

    print(f"USER: {prompt}\n")
    audio_bytes = 0
    transcript = ""
    try:
        async for event in runner.run_live(
            user_id=USER, session_id="live1",
            live_request_queue=queue, run_config=config,
        ):
            if event.content and event.content.parts:
                for p in event.content.parts:
                    if p.inline_data and p.inline_data.data:
                        audio_bytes += len(p.inline_data.data)
            ot = event.output_transcription
            if ot and ot.text:
                if ot.finished:
                    transcript = ot.text  # consolidated final transcript
                else:
                    print(ot.text, end="", flush=True)  # words as they stream
            if event.turn_complete:
                break
    except Exception as e:
        print(f"❌ Live API error: {type(e).__name__}: {str(e)[:200]}")
        print()
        print("The Live API is preview-tier; transient server errors happen. Re-run")
        print("the cell, and see DEMOS_BROKEN.md in the repo for the current state.")
    finally:
        queue.close()

    print(f"\n\n[audio] {audio_bytes:,} bytes of 24kHz PCM received")
    print(f"[transcript] {transcript.strip()}")

await live_audio_turn("Say hi and tell me one fun fact about octopuses.")


That's the whole API shape:

1. Create a `LiveRequestQueue`. This is where you feed inputs from whatever source — keyboard, microphone, etc.
2. Call `runner.run_live(...)` with the queue and a `RunConfig`. Audio-native models want `response_modalities=["AUDIO"]`; add `output_audio_transcription` to stream the words.
3. Iterate `async for event in ...` — audio arrives as `inline_data` parts, the words as `event.output_transcription` (streaming chunks, then one consolidated chunk with `finished=True`), until `event.turn_complete`.
4. Finally, `queue.close()` to end the session cleanly.

For real voice input, the queue's `send_realtime(...)` method takes raw PCM audio bytes (16kHz, mono, 16-bit) instead of `types.Content`. You'd feed it from a microphone library like `pyaudio`, `sounddevice`, or a browser WebSocket.


# Voice Activity Detection + Interruption

Two features that make Live feel human-like, both handled server-side by Gemini:

- **Voice Activity Detection (VAD)**: Gemini decides when the user has finished speaking and starts responding. You don't have to say "ok, I'm done"; silence triggers response.
- **Interruption**: If the user starts talking while Gemini is mid-response, Gemini stops talking immediately and listens to the new input. The natural "wait, actually..." pattern humans use.

Both are on by default. You can tune VAD sensitivity via `realtime_input_config` in `RunConfig`:

```python
from google.genai.types import RealtimeInputConfig, AutomaticActivityDetection

config = RunConfig(
    response_modalities=["AUDIO"],
    realtime_input_config=RealtimeInputConfig(
        automatic_activity_detection=AutomaticActivityDetection(
            silence_duration_ms=1000,     # wait 1s of silence before responding
            prefix_padding_ms=200,        # grab 200ms before detected speech
        )
    ),
)
```

Lower `silence_duration_ms` → snappier but more false triggers. Higher → more natural-feeling but noticeable delay. 1000ms is a sensible default for conversation; 500ms for command-style interfaces where users know what they'll say.

# A Working Production Architecture — Sketch

What a real Live-voice product looks like at the architecture level:

```
Browser (user)                  Your backend                Gemini
──────────────                  ──────────────              ──────
Microphone input  ──WebSocket──▶  ADK agent + runner.run_live()
                                       │
                                       ├── LiveRequestQueue ◀── audio chunks
                                       │
                                       │                   WebSocket to
                                       │                    Live API
                                       │
                                       ◀── event.audio ──▶  audio chunks
                                       │
Audio playback   ◀──WebSocket──  stream response audio back
                                 (+ interruption signals)
```

Each user turn is pushed to `LiveRequestQueue.send_realtime(...)` as 16kHz PCM chunks. Response events with `event.content.parts[].inline_data` contain response audio (24kHz PCM for Gemini). Your backend decodes them and writes them out to the browser's audio sink.

None of this lives in a notebook — the full path needs a browser with microphone access, a WebSocket server, and audio-level libraries. What lives in the notebook is the ADK-side of the API contract. For a full-stack version, start from the [adk-samples](https://github.com/google/adk-samples) voice-agent example.

**Pricing**: audio is billed by minute, not by token. Gemini Flash Live (preview, free tier) is free-tier-eligible up to quota; `gemini-2.5-flash-native-audio-latest` is ~$0.012 per minute of audio in/out on paid tier.

# When Live Is the Right Tool

Live is specifically for **real-time conversational voice**. It is not the right tool for:

- **Transcription**. Use a dedicated speech-to-text service; feed the text to a regular agent.
- **TTS playback of pre-computed answers.** Use a dedicated TTS service.
- **Non-interactive voice.** An agent that narrates a pre-written script doesn't need Live.

Live shines when you need:

- **Sub-second response latency** — users notice delays above ~300ms in conversation.
- **Interruption handling** — users will talk over you; the natural conversational pattern needs it.
- **Open-session continuous dialog** — not turn-based chat but fluid back-and-forth.

For everything else, `run_async()` with text-in/text-out plus separate STT/TTS is simpler, cheaper, and more controllable.

**Google's position at April 2026**: Live is the single most jaw-dropping capability Gemini ships that no competitor replicates cleanly. OpenAI's Realtime API is the closest alternative; Anthropic has no direct equivalent. If voice-first conversational agents are your product, Gemini Live is where you build.

# Your Turn

Some of these depend on your environment cooperating with the Live API. Skip what doesn't apply.

1. **Discover Live models on your key.** The first code cell lists them. How many are listed — zero, one, more? Which of those require a paid tier?
2. **Text-mode rejection.** Change `response_modalities` to `["TEXT"]` and re-run. What error does the server return? (Expect 1007 — current live models are audio-native.)
3. **Transcript streaming.** Print every `output_transcription` chunk with its `finished` flag as it arrives. How does the partial text build up, and what does the final consolidated chunk look like?
4. **Rate-limit exploration.** Run multiple concurrent Live sessions in separate notebook cells. What rate limits do you hit on your key's tier?

# Key Takeaways

- **Live is the most differentiated Gemini capability.** Bidirectional audio, VAD, interruption, sub-second latency. No competitor replicates it cleanly as of mid-2026.
- **Three primitives:** `LiveRequestQueue` (input), `Runner.run_live()` (event generator), `RunConfig(response_modalities=[...])` (output shape).
- **API shape**: push inputs into the queue, async-for over events, `turn_complete` ends a turn, `queue.close()` ends the session.
- **VAD + interruption** handled server-side by Gemini; tunable via `RealtimeInputConfig.automatic_activity_detection`.
- **Production architecture** is browser-microphone → WebSocket → backend ADK agent → Gemini Live API → response audio back. Notebook can't do the full loop.
- **Preview-tier fragility is real.** Live spent spring 2026 throwing transient 1011 server errors before Google patched the endpoint in July; it runs cleanly now, but code defensively.
- **Model names move**: check the catalog. As of this writing: `gemini-3.1-flash-live-preview` (free-tier-available), `gemini-2.5-flash-native-audio-latest` (paid, production-grade).
- **Use Live only for real-time conversational voice.** For STT, TTS, or non-interactive use, simpler services.

# Next up — M14: A2A protocol

The final module. The side step into **agent-to-agent** communication — the protocol Google donated to the Linux Foundation in June 2025 that's now the industry standard for agents-talking-to-agents across vendors. 30-minute block: the four nouns (Card, Task, Message, Artifact), the wow demo (ADK orchestrator calling a LangGraph currency-converter specialist over A2A), and the gotchas worth knowing — because A2A is still preview-tier with its own version-alignment issues.